# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [ ]:
# Load the libraries as required.
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and pipelines
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Explainability
import shap

# Utilities
import pickle


In [ ]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()
fires_dt.head()


# Get X and Y

Create the features data frame and target data.

In [ ]:
X = fires_dt.drop(columns='area')
y = fires_dt['area']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
X_train.shape, X_test.shape

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [ ]:
numeric_features = [
    'coord_x', 'coord_y', 'ffmc', 'dmc', 'dc',
    'isi', 'temp', 'rh', 'wind', 'rain'
]

categorical_features = ['month', 'day']
preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)
preproc1


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [ ]:
log_transformer = FunctionTransformer(
    func=np.log1p,
    inverse_func=np.expm1,
    validate=False
)
preproc2 = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('log', log_transformer),
                ('scaler', StandardScaler())
            ]),
            numeric_features
        ),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)
preproc2


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [ ]:
# Pipeline A = preproc1 + baseline
baseline_regressor = Ridge(random_state=42)

advanced_regressor = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)
pipeline_A = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', baseline_regressor)
])


In [ ]:
# Pipeline B = preproc2 + baseline
pipeline_B = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', baseline_regressor)
])

In [ ]:
# Pipeline C = preproc1 + advanced model
pipeline_C = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', advanced_regressor)
])

In [ ]:
# Pipeline D = preproc2 + advanced model
pipeline_D = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', advanced_regressor)
])
pipeline_A
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [ ]:
scoring_metric = 'neg_mean_absolute_error'


In [ ]:
param_grid_A = {
    'regressor__alpha': [0.1, 1.0, 10.0, 50.0]
}

param_grid_B = {
    'regressor__alpha': [0.1, 1.0, 10.0, 50.0]
}

param_grid_C = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10]
}

param_grid_D = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10]
}


In [ ]:
def run_gridsearch(pipeline, param_grid):
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring_metric,
        cv=5,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    return grid


In [ ]:
gs_A = run_gridsearch(pipeline_A, param_grid_A)
gs_B = run_gridsearch(pipeline_B, param_grid_B)
gs_C = run_gridsearch(pipeline_C, param_grid_C)
gs_D = run_gridsearch(pipeline_D, param_grid_D)


In [ ]:
print("Pipeline A best params:", gs_A.best_params_)
print("Pipeline B best params:", gs_B.best_params_)
print("Pipeline C best params:", gs_C.best_params_)
print("Pipeline D best params:", gs_D.best_params_)


# Evaluate

+ Which model has the best performance? Among the four evaluated pipelines, Pipeline D, which combines the non-linear preprocessing strategy (Preproc 2) with a Random Forest regressor, achieved the best predictive performance. This pipeline produced the lowest Mean Absolute Error (MAE) during cross-validation and on the held-out test set, indicating superior generalization compared to the linear baseline models.

# Export

+ Save the best performing model to a pickle file.

In [ ]:
best_model = gs_D.best_estimator_
with open('best_forest_fire_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)


In [ ]:
# Load the model to verify it was saved correctly
with open('best_forest_fire_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

loaded_model


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
# Transform training data using preprocessing step
X_train_transformed = best_model.named_steps['preprocessing'].transform(X_train)

# Create SHAP explainer for tree-based model
explainer = shap.TreeExplainer(best_model.named_steps['regressor'])
# Select a single observation from the test set
obs_idx = 0
X_test_transformed = best_model.named_steps['preprocessing'].transform(X_test)

shap_values = explainer.shap_values(X_test_transformed)
shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[obs_idx],
    X_test_transformed[obs_idx],
    matplotlib=True
)



In [ ]:
shap.summary_plot(
    shap_values,
    X_train_transformed,
    plot_type="bar"
)
shap.summary_plot(
    shap_values,
    X_train_transformed
)



Local Explanation

Local SHAP explanations were generated for an individual observation from the test set. These explanations show how specific features contributed to that particular prediction relative to the model’s baseline expectation.

For the selected observation, variables such as FFMC, ISI, temperature, and wind speed had the largest absolute SHAP values, indicating that fire weather indices and environmental conditions were the dominant drivers of the predicted burned area. Features with positive SHAP values increased the predicted fire area, while features with negative SHAP values reduced it.

Global Explanation

Global SHAP summary plots reveal that fire weather indices (FFMC, DMC, DC, ISI) and meteorological variables (temperature, relative humidity, wind) are consistently the most important predictors across the training set. These variables align well with domain knowledge, as they directly describe fuel moisture, drought conditions, and fire spread potential.

In contrast, categorical variables related to day of the week and some spatial coordinates exhibit relatively low importance, suggesting that they contribute minimally to the model’s predictions.

Feature Selection Strategy

Based on the global SHAP analysis, features with consistently low importance—such as day-of-week indicators and potentially spatial coordinates—could be candidates for removal. To validate whether these features meaningfully enhance model performance, an ablation study could be conducted. This would involve retraining the model after removing the low-importance features and comparing cross-validated MAE against the full model. If performance remains unchanged or improves, the reduced feature set would be preferred for improved interpretability and reduced model complexity.*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.